# Verify YOLO joints and action labels

Use this notebook after running `build_action_joint_dataset.py`. It validates the raw long-form detections, selects the largest detected person in each frame using the shared model preprocessing, overlays that skeleton on the original CFR video frame, and shows the assigned action class.

The browser uses the saved CSV coordinates rather than running YOLO again, so what you see is exactly what will be used by the training dataset.

## 1. Generate the dataset if needed

Run this from the repository root in a terminal:

```powershell
conda run -n muay-thai python dataset\build_action_joint_dataset.py
```

Existing CSV files are skipped. Add `--overwrite` when you intentionally want to regenerate them.

In [ ]:
from pathlib import Path
import sys

import cv2
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

def find_repository_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "dataset").is_dir() and (candidate / "models").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the Muay-ThAI repository root")


ROOT_DIR = find_repository_root(Path.cwd().resolve())
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from models.action_detection.config import (
    CLASS_COLORS,
    SKELETON_EDGES,
    validate_task_labels,
)
from models.action_detection.preprocessing import (
    select_largest_person,
    validate_raw_schema,
)
from models.yolo.config import YOLO_KEYPOINT_NAMES

CLASSIFICATION_TASK = "striking"  # Change to "striking" or "guard" as needed.
JOINTS_DIR = (
    ROOT_DIR / "dataset" / "jointswithactionlabels" / CLASSIFICATION_TASK
)
VIDEO_DIR = ROOT_DIR / "media" / "videos" / "30fps"

print(f"Repository: {ROOT_DIR}")
print(f"Joint CSVs: {JOINTS_DIR}")
print(f"Videos:     {VIDEO_DIR}")

In [ ]:
csv_paths = sorted(JOINTS_DIR.glob("*_joints_labels.csv"))
if not csv_paths:
    raise FileNotFoundError(
        f"No joint CSV files found in {JOINTS_DIR}. "
        "Run dataset/build_action_joint_dataset.py first."
    )

raw_datasets = []
for csv_path in csv_paths:
    raw_dataset = pd.read_csv(csv_path)
    validate_raw_schema(raw_dataset, csv_path)
    raw_datasets.append(raw_dataset)

raw_frames = pd.concat(raw_datasets, ignore_index=True)
validate_task_labels(
    CLASSIFICATION_TASK,
    raw_frames["action_label"].astype(str),
    source=str(JOINTS_DIR),
)
raw_frames = raw_frames.sort_values(
    ["video_id", "frame_index", "detection_index"],
    kind="stable",
).reset_index(drop=True)

duplicate_detections = raw_frames.duplicated(
    ["video_id", "frame_index", "detection_index"]
).sum()
if duplicate_detections:
    raise ValueError(
        f"Found {duplicate_detections} duplicate frame/detection rows"
    )

frames = select_largest_person(raw_frames).reset_index(drop=True)
duplicate_selected_frames = frames.duplicated(
    ["video_id", "frame_index"]
).sum()
if duplicate_selected_frames:
    raise ValueError(
        f"Found {duplicate_selected_frames} duplicate selected frames"
    )

print(
    f"Loaded {len(raw_frames):,} raw detection rows and selected "
    f"{len(frames):,} frames from {len(csv_paths)} CSV files"
)
display(raw_frames.head(3))
display(frames.head(3))

## 2. Dataset summary

The summary uses one largest-person row per frame. `missing_poses` is worth checking after a complete extraction. Background frames may legitimately have no person, but long gaps inside an action segment usually need inspection.

In [ ]:
summary = (
    frames.groupby("video_id")
    .agg(
        frames=("frame_index", "nunique"),
        first_frame=("frame_index", "min"),
        last_frame=("frame_index", "max"),
        detected_poses=("pose_detected", "sum"),
    )
)
summary["missing_poses"] = summary["frames"] - summary["detected_poses"]
summary["missing_pose_percent"] = (
    100 * summary["missing_poses"] / summary["frames"]
).round(2)

display(summary)
display(
    frames.groupby(["video_id", "action_label"])
    .size()
    .rename("frames")
    .unstack(fill_value=0)
)

## 3. Interactive skeleton viewer

- Choose a video and move the frame slider.
- Increase the confidence threshold to hide uncertain joints.
- Empty skeletons are displayed normally so missing detections remain visible.
- The title and frame overlay show the saved manual action class.

In [ ]:
JOINT_NAMES = tuple(YOLO_KEYPOINT_NAMES)


def read_video_frame(video_id: str, frame_index: int) -> np.ndarray:
    """
    Reads a specific frame from a video file and returns it as an RGB image.
    """
    video_path = VIDEO_DIR / f"{video_id}.mp4"
    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        raise FileNotFoundError(f"Could not open {video_path}")

    capture.set(cv2.CAP_PROP_POS_FRAMES, int(frame_index))
    success, frame_bgr = capture.read()
    capture.release()
    if not success:
        raise ValueError(f"Could not decode {video_id} frame {frame_index}")
    return cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)


def draw_saved_pose(
    image: np.ndarray,
    row: pd.Series,
    confidence_threshold: float,
) -> np.ndarray:
    """
    Draws the pose skeleton and bounding box on the given image based on the
    provided row of pose data. Only joints with confidence above the specified threshold are drawn. The action label, frame index, pose detection status,
    number of people detected, and selected detection index are displayed as text on the image.
    """
    output = image.copy()
    height, width = output.shape[:2]
    action_label = str(row["action_label"])
    color = CLASS_COLORS.get(action_label, (255, 200, 0))
    points = {}

    for joint_name in JOINT_NAMES:
        confidence = row.get(f"{joint_name}_confidence", np.nan)
        x = row.get(f"{joint_name}_x_px", np.nan)
        y = row.get(f"{joint_name}_y_px", np.nan)
        if pd.isna(confidence) or pd.isna(x) or pd.isna(y):
            continue
        if float(confidence) < confidence_threshold:
            continue
        point = (
            int(np.clip(float(x), 0, width - 1)),
            int(np.clip(float(y), 0, height - 1)),
        )
        points[joint_name] = point

    for start_name, end_name in SKELETON_EDGES:
        if start_name in points and end_name in points:
            cv2.line(output, points[start_name], points[end_name], color, 3)
    for point in points.values():
        cv2.circle(output, point, 5, (255, 255, 255), -1)
        cv2.circle(output, point, 5, color, 2)

    if not pd.isna(row.get("bbox_x1_px", np.nan)):
        x1 = int(np.clip(float(row["bbox_x1_px"]), 0, width - 1))
        y1 = int(np.clip(float(row["bbox_y1_px"]), 0, height - 1))
        x2 = int(np.clip(float(row["bbox_x2_px"]), 0, width - 1))
        y2 = int(np.clip(float(row["bbox_y2_px"]), 0, height - 1))
        cv2.rectangle(output, (x1, y1), (x2, y2), color, 2)

    status = "pose detected" if int(row["pose_detected"]) else "NO POSE"
    selected_index = row.get("detection_index", np.nan)
    selected_text = (
        "none" if pd.isna(selected_index) else str(int(selected_index))
    )
    text = (
        f"{action_label} | frame {int(row['label_frame'])} "
        f"| {status} | people {int(row['people_detected'])} "
        f"| selected {selected_text}"
    )
    cv2.rectangle(output, (0, 0), (min(width, 900), 48), (0, 0, 0), -1)
    cv2.putText(
        output,
        text,
        (12, 32),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        color,
        2,
        cv2.LINE_AA,
    )
    return output


def show_pose_frame(
    video_id: str,
    frame_index: int,
    confidence_threshold: float = 0.25,
) -> None:
    """
    Displays a specific frame from a video with the pose skeleton and bounding box overlaid based on the corresponding row of pose data. The function retrieves the frame from the video, draws the pose using the provided confidence threshold, and shows the result using matplotlib.
    """
    matching = frames[
        (frames["video_id"] == video_id)
        & (frames["frame_index"] == int(frame_index))
    ]
    if matching.empty:
        print(f"No dataset row for {video_id}, frame {frame_index}")
        return

    row = matching.iloc[0]
    image = read_video_frame(video_id, int(frame_index))
    overlay = draw_saved_pose(image, row, confidence_threshold)

    plt.figure(figsize=(14, 8))
    plt.imshow(overlay)
    plt.title(
        f"{video_id} — frame_index={int(frame_index)}, "
        f"class={row['action_label']}"
    )
    plt.axis("off")
    plt.show()

In [ ]:
video_ids = sorted(frames["video_id"].unique())
video_widget = widgets.Dropdown(
    options=video_ids,
    value=video_ids[0],
    description="Video:",
    layout=widgets.Layout(width="600px"),
)


def frame_bounds(video_id: str) -> tuple[int, int]:
    video_frames = frames.loc[frames["video_id"] == video_id, "frame_index"]
    return int(video_frames.min()), int(video_frames.max())


initial_min, initial_max = frame_bounds(video_widget.value)
frame_widget = widgets.IntSlider(
    value=initial_min,
    min=initial_min,
    max=initial_max,
    step=1,
    description="Frame:",
    continuous_update=False,
    layout=widgets.Layout(width="900px"),
)
confidence_widget = widgets.FloatSlider(
    value=0.25,
    min=0.0,
    max=1.0,
    step=0.05,
    description="Joint conf:",
    continuous_update=False,
    readout_format=".2f",
    layout=widgets.Layout(width="500px"),
)


def update_frame_slider(change) -> None:
    minimum, maximum = frame_bounds(change["new"])
    frame_widget.min = minimum
    frame_widget.max = maximum
    frame_widget.value = minimum


video_widget.observe(update_frame_slider, names="value")
viewer_output = widgets.interactive_output(
    show_pose_frame,
    {
        "video_id": video_widget,
        "frame_index": frame_widget,
        "confidence_threshold": confidence_widget,
    },
)

display(widgets.VBox([video_widget, frame_widget, confidence_widget]), viewer_output)

## 4. Review samples from one class

This view chooses evenly spaced examples rather than random examples, making repeated checks reproducible.

In [ ]:
def show_class_samples(
    action_label: str,
    video_id: str | None = None,
    sample_count: int = 6,
    confidence_threshold: float = 0.25,
) -> None:
    """
    Displays a grid of sample frames from the dataset that match the specified action label and optional video ID. The function selects a specified number of frames evenly spaced across the available candidates, overlays the pose skeleton and bounding box on each frame based on the provided confidence threshold, and shows the results in a matplotlib figure.
    """
    candidates = frames[frames["action_label"] == action_label]
    if video_id is not None:
        candidates = candidates[candidates["video_id"] == video_id]
    if candidates.empty:
        print("No matching frames")
        return

    positions = np.linspace(
        0,
        len(candidates) - 1,
        min(sample_count, len(candidates)),
        dtype=int,
    )
    selected = candidates.iloc[positions]
    columns = 3
    rows = int(np.ceil(len(selected) / columns))
    figure, axes = plt.subplots(rows, columns, figsize=(18, 6 * rows))
    axes = np.atleast_1d(axes).ravel()

    for axis, (_, row) in zip(axes, selected.iterrows()):
        image = read_video_frame(row["video_id"], int(row["frame_index"]))
        overlay = draw_saved_pose(image, row, confidence_threshold)
        axis.imshow(overlay)
        axis.set_title(f"{row['video_id']} — frame {int(row['label_frame'])}")
        axis.axis("off")
    for axis in axes[len(selected):]:
        axis.axis("off")

    figure.suptitle(f"Class samples: {action_label}", fontsize=18)
    figure.tight_layout()
    plt.show()


# Show one non-background class across the complete selected dataset.
example_labels = [label for label in sorted(frames["action_label"].unique())]
if "_example_label_index" not in globals():
    _example_label_index = -1
_example_label_index = (_example_label_index + 1) % len(example_labels)
example_label = example_labels[_example_label_index]


show_class_samples(example_label, sample_count=6)

## 5. Review horizontal-flip augmentation

These cells use the production `load_pose_sequences()` normalization and `horizontal_flip_pose_features()` augmentation. The original and mirrored normalized skeletons should be exact horizontal reflections, while the action label remains unchanged. Blue joints are anatomically left, red joints are anatomically right, and gold joints are unpaired.

Use both views below:

1. The interactive frame viewer compares the saved video pose, the expected flipped RGB frame, and the exact normalized features used for training.
2. The motion strip checks several frames from one class so that the complete action remains temporally coherent after mirroring.

A good result has no distortion or sudden side changes, preserves missing-joint patterns and confidence, and reports a zero (or tiny floating-point) round-trip error after flipping twice.

In [ ]:
import models.action_detection.preprocessing as action_preprocessing

FEATURE_CHANNELS = action_preprocessing.FEATURE_CHANNELS
horizontal_flip_pose_features = (
    action_preprocessing.horizontal_flip_pose_features
)
load_pose_sequences = action_preprocessing.load_pose_sequences


AUGMENTATION_CONFIDENCE_THRESHOLD = 0.25
AUGMENTATION_COORDINATE_CLIP = 5.0
pose_sequences = load_pose_sequences(
    JOINTS_DIR,
    classification_task=CLASSIFICATION_TASK,
    confidence_threshold=AUGMENTATION_CONFIDENCE_THRESHOLD,
    coordinate_clip=AUGMENTATION_COORDINATE_CLIP,
)
mirrored_pose_features = {
    video_id: horizontal_flip_pose_features(sequence.features)
    for video_id, sequence in pose_sequences.items()
}
round_trip_errors = {
    video_id: float(
        np.max(
            np.abs(
                horizontal_flip_pose_features(mirrored_pose_features[video_id])
                - sequence.features
            )
        )
    )
    for video_id, sequence in pose_sequences.items()
}


def joint_plot_color(joint_name: str) -> str:
    if joint_name.startswith("left_"):
        return "#1677ff"
    if joint_name.startswith("right_"):
        return "#e53935"
    return "#d49b00"


def draw_normalized_pose(
    axis,
    flat_features: np.ndarray,
    title: str,
    *,
    show_joint_names: bool = False,
) -> None:
    pose = flat_features.reshape(len(JOINT_NAMES), len(FEATURE_CHANNELS))
    valid = pose[:, 3] >= 0.5
    points = {
        joint_name: pose[joint_index]
        for joint_index, joint_name in enumerate(JOINT_NAMES)
        if valid[joint_index]
    }

    for start_name, end_name in SKELETON_EDGES:
        if start_name in points and end_name in points:
            start = points[start_name]
            end = points[end_name]
            axis.plot(
                [start[0], end[0]],
                [start[1], end[1]],
                color="#666666",
                linewidth=2,
                zorder=1,
            )

    for joint_name, point in points.items():
        confidence = float(point[2])
        axis.scatter(
            point[0],
            point[1],
            s=35 + 75 * confidence,
            color=joint_plot_color(joint_name),
            edgecolor="white",
            linewidth=0.8,
            alpha=max(0.3, confidence),
            zorder=2,
        )
        if show_joint_names:
            axis.annotate(
                joint_name.replace("left_", "L_").replace("right_", "R_"),
                (point[0], point[1]),
                xytext=(4, 4),
                textcoords="offset points",
                fontsize=7,
            )

    limit = AUGMENTATION_COORDINATE_CLIP + 0.25
    axis.axvline(0.0, color="#bbbbbb", linewidth=1, linestyle="--")
    axis.axhline(0.0, color="#dddddd", linewidth=1, linestyle=":")
    axis.set_xlim(-limit, limit)
    axis.set_ylim(limit, -limit)
    axis.set_aspect("equal", adjustable="box")
    axis.set_xlabel("x_body (torso lengths)")
    axis.set_ylabel("y_body (torso lengths)")
    axis.set_title(f"{title} — {len(points)}/{len(JOINT_NAMES)} valid joints")
    axis.grid(alpha=0.15)


def sequence_position(video_id: str, frame_index: int) -> int:
    positions = np.flatnonzero(
        pose_sequences[video_id].frame_indices == int(frame_index)
    )
    if not len(positions):
        raise ValueError(f"No frame {frame_index} in {video_id}")
    return int(positions[0])


def show_horizontal_flip_frame(
    video_id: str,
    frame_index: int,
    show_joint_names: bool = False,
) -> None:
    sequence = pose_sequences[video_id]
    position = sequence_position(video_id, frame_index)
    action_label = str(sequence.labels[position])
    image = read_video_frame(video_id, frame_index)
    matching = frames[
        (frames["video_id"] == video_id)
        & (frames["frame_index"] == int(frame_index))
    ]
    original_image = (
        draw_saved_pose(
            image,
            matching.iloc[0],
            AUGMENTATION_CONFIDENCE_THRESHOLD,
        )
        if not matching.empty
        else image
    )

    figure, axes = plt.subplots(1, 4, figsize=(24, 6))
    axes[0].imshow(original_image)
    axes[0].set_title("Original saved pose")
    axes[0].axis("off")
    axes[1].imshow(np.ascontiguousarray(image[:, ::-1]))
    axes[1].set_title("Expected horizontal RGB reflection")
    axes[1].axis("off")
    draw_normalized_pose(
        axes[2],
        sequence.features[position],
        "Normalized original",
        show_joint_names=show_joint_names,
    )
    draw_normalized_pose(
        axes[3],
        mirrored_pose_features[video_id][position],
        "Training augmentation",
        show_joint_names=show_joint_names,
    )
    figure.suptitle(
        f"{video_id} | frame {frame_index} | unchanged label={action_label} | "
        f"double-flip max error={round_trip_errors[video_id]:.3g}",
        fontsize=16,
    )
    figure.tight_layout()
    plt.show()


display(pd.Series(round_trip_errors, name="double_flip_max_abs_error"))

In [ ]:
augmentation_video_widget = widgets.Dropdown(
    options=sorted(pose_sequences),
    description="Video:",
    layout=widgets.Layout(width="600px"),
)
initial_sequence = pose_sequences[augmentation_video_widget.value]
augmentation_frame_widget = widgets.IntSlider(
    value=int(initial_sequence.frame_indices[0]),
    min=int(initial_sequence.frame_indices[0]),
    max=int(initial_sequence.frame_indices[-1]),
    step=1,
    description="Frame:",
    continuous_update=False,
    layout=widgets.Layout(width="900px"),
)
augmentation_names_widget = widgets.Checkbox(
    value=False,
    description="Show joint names",
)


def update_augmentation_frame_bounds(change) -> None:
    sequence = pose_sequences[change["new"]]
    augmentation_frame_widget.min = int(sequence.frame_indices[0])
    augmentation_frame_widget.max = int(sequence.frame_indices[-1])
    augmentation_frame_widget.value = int(sequence.frame_indices[0])


augmentation_video_widget.observe(
    update_augmentation_frame_bounds,
    names="value",
)
augmentation_frame_output = widgets.interactive_output(
    show_horizontal_flip_frame,
    {
        "video_id": augmentation_video_widget,
        "frame_index": augmentation_frame_widget,
        "show_joint_names": augmentation_names_widget,
    },
)
display(
    widgets.VBox(
        [
            augmentation_video_widget,
            augmentation_frame_widget,
            augmentation_names_widget,
        ]
    ),
    augmentation_frame_output,
)

## Manual review checklist

Pay particular attention to:

1. Class transitions: inspect frames immediately before and after each change between task labels.
2. Subject selection: confirm the bounding box follows the intended athlete when multiple people are visible.
3. Pose failures: look for missing or low-confidence task-relevant joints during fast movement.
4. Temporal consistency: watch for the skeleton suddenly jumping to another person.
5. Missing poses inside action segments: these are more concerning than missing poses in `background`.